In [ ]:
# splitting into train test split
# get pandas
import pandas as pd

# get sk learn
from sklearn.model_selection import train_test_split

# get keras
#!pip install tensorflow keras


In [ ]:
# load the cleaned dataset into a pandas dataframe
df = pd.read_csv('clean_data.csv')

X = df['Narrative Text']
y = df['Event Label']

In [ ]:
# split data into train and test (70/30) then I'll split the test into a dev and test 50/50
#
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# now splitting test up into 50/50 validation test

X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=42)

# now the data is 70/15/15 with a static seed of 42

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

# data split proportionally per label (original proportions matched)
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(11125,)
(2384,)
(2384,)
Event Label
Equipment Problem    0.545618
Deviation            0.247011
ATC Issue            0.115506
Conflict             0.051685
Inflight Event       0.025798
Ground Event         0.014382
Name: proportion, dtype: float64
Event Label
Equipment Problem    0.545302
Deviation            0.246644
ATC Issue            0.115772
Conflict             0.052013
Inflight Event       0.026007
Ground Event         0.014262
Name: proportion, dtype: float64
Event Label
Equipment Problem    0.545721
Deviation            0.247064
ATC Issue            0.115352
Conflict             0.051594
Inflight Event       0.025587
Ground Event         0.014681
Name: proportion, dtype: float64


Using TFIDF for vectors for each narrative

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# obj for prop
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)

# learn/fit and turn to vectors (trans)
X_train_tfidf = vectorizer.fit_transform(X_train)

# just turn to vectors using setup from training set only trans
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_val_tfidf.shape)
print(X_test_tfidf.shape)

(11125, 10000)
(2384, 10000)
(2384, 10000)


Converting the labels to vectors
0 -> 6 ints then
[0, 0, 0, 1, 0, 0] vectors

          ^
          |
    current category

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

print("before label encoder")
print(y_train.head())
print(y_val.head())
print(y_test.head())
print(' ')


# obj for prop
label_vectors = LabelEncoder()

# learn/fit on train
y_train_labels = label_vectors.fit_transform(y_train)

# transform the val and test
y_val_labels = label_vectors.transform(y_val)
y_test_labels = label_vectors.transform(y_test)

print("after label encoder")
print(y_train_labels)
print(y_val_labels)
print(y_test_labels)
print(' ')

y_train_categorical = to_categorical(y_train_labels)
y_val_categorical = to_categorical(y_val_labels)
y_test_categorical = to_categorical(y_test_labels)

print("after categorical conversion")
print(y_train_categorical)
print(y_val_categorical)
print(y_test_categorical)
print(' ')



before label encoder
9892     Equipment Problem
1678             Deviation
13219    Equipment Problem
9576             Deviation
9160     Equipment Problem
Name: Event Label, dtype: object
9085             Deviation
8852     Equipment Problem
14561            Deviation
15335    Equipment Problem
4138             Deviation
Name: Event Label, dtype: object
15030    Equipment Problem
12960            Deviation
492               Conflict
9428             Deviation
4829     Equipment Problem
Name: Event Label, dtype: object
 
after label encoder
[3 2 3 ... 1 3 3]
[2 3 2 ... 3 3 3]
[3 2 1 ... 3 3 2]
 
after categorical conversion
[[0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 ...
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]]
[[0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 ...
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]]
[[0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0

In [ ]:
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense
from tensorflow.keras import models, layers
# early stopping to prevent overfitting
from tensorflow.keras.callbacks import EarlyStopping

network = models.Sequential()

# tfidf vectors have 42817 features
network.add(layers.Dense(128, activation='relu', input_shape=(10000,)))

# 6 classes for classification
network.add(layers.Dense(6, activation='softmax'))


network.compile(optimizer='rmsprop', loss='categorical_crossentropy',metrics=['accuracy'])

# train with tfidf vectors for Narrative and categorical vectors for each event label
# validation added to get extra stats during training epochs
# tried 20 epochs and it overfit like crazy, im going to try early stopping now

# 3 epochs of no improvement before pulling the plug and reverting weights to lowest val loss
stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
# added early stop
network.fit(X_train_tfidf, y_train_categorical, epochs=20, batch_size=128, validation_data=(X_val_tfidf, y_val_categorical), callbacks=[stop])

Epoch 1/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.5955 - loss: 1.1359 - val_accuracy: 0.7202 - val_loss: 0.8103
Epoch 2/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.7744 - loss: 0.6924 - val_accuracy: 0.7966 - val_loss: 0.6150
Epoch 3/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.8184 - loss: 0.5478 - val_accuracy: 0.8091 - val_loss: 0.5458
Epoch 4/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8450 - loss: 0.4702 - val_accuracy: 0.8192 - val_loss: 0.5150
Epoch 5/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8605 - loss: 0.4162 - val_accuracy: 0.8184 - val_loss: 0.5011
Epoch 6/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.8755 - loss: 0.3731 - val_accuracy: 0.8188 - val_loss: 0.5016
Epoch 7/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8893 - loss: 0.3359 - val_accuracy: 0.8201 - val_loss: 0.4983
Epoch 8/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9014 - loss: 0.3017 - val_accuracy: 0.8205 - v

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# testing on test
network.evaluate(X_test_tfidf, y_test_categorical)

# evaluating before applying the inverse frequency weighting recommendation

predictions = network.predict(X_test_tfidf)
# find max value per output vector
y_pred = np.argmax(predictions, axis=1)
# table output showing per class stats
print(f"{classification_report(y_test_labels, y_pred, target_names=label_vectors.classes_)}")






75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8234 - loss: 0.4882
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
                   precision    recall  f1-score   support

        ATC Issue       0.69      0.72      0.71       275
         Conflict       0.70      0.63      0.66       123
        Deviation       0.73      0.74      0.74       589
Equipment Problem       0.91      0.93      0.92      1301
     Ground Event       1.00      0.09      0.16        35
   Inflight Event       0.72      0.56      0.63        61

         accuracy                           0.82      2384
        macro avg       0.79      0.61      0.64      2384
     weighted avg       0.82      0.82      0.82      2384



In [ ]:
from sklearn.utils.class_weight import compute_class_weight
# inverse frequency waiting trial

# get the new weights for each class applying inverse frequency weighting (balanced)
new_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)

# keras requires a dictionary so the array needs to be converted
class_weight_dict = {}
for i in range(len(new_weights)):
    class_weight_dict[i] = new_weights[i]

# checking what weight each class got
#
for i, name in enumerate(label_vectors.classes_):
      print(f"{name:20s} {class_weight_dict[i]:.3f}")



ATC Issue            1.443
Conflict             3.225
Deviation            0.675
Equipment Problem    0.305
Ground Event         11.589
Inflight Event       6.461


In [ ]:
# remaking the model w new weights

network = models.Sequential()

# tfidf vectors have 42817 features
network.add(layers.Dense(128, activation='relu', input_shape=(10000,)))

# 6 classes for classification
network.add(layers.Dense(6, activation='softmax'))


network.compile(optimizer='rmsprop', loss='categorical_crossentropy',metrics=['accuracy'])

# train with tfidf vectors for Narrative and categorical vectors for each event label
# validation added to get extra stats during training epochs
# tried 20 epochs and it overfit like crazy, im going to try early stopping now

# 3 epochs of no improvement before pulling the plug and reverting weights to lowest val loss
stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
# added early stop
network.fit(X_train_tfidf, y_train_categorical, epochs=20, batch_size=128, validation_data=(X_val_tfidf, y_val_categorical), callbacks=[stop], class_weight=class_weight_dict)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.6597 - loss: 1.5476 - val_accuracy: 0.7387 - val_loss: 1.1581
Epoch 2/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.7465 - loss: 1.0259 - val_accuracy: 0.7508 - val_loss: 0.8272
Epoch 3/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.7686 - loss: 0.7404 - val_accuracy: 0.7617 - val_loss: 0.7129
Epoch 4/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.7920 - loss: 0.5911 - val_accuracy: 0.7664 - val_loss: 0.6513
Epoch 5/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.8152 - loss: 0.4933 - val_accuracy: 0.7752 - val_loss: 0.6182
Epoch 6/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8377 - loss: 0.4180 - val_accuracy: 0.7794 - val_loss: 0.6119
Epoch 7/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.8563 - loss: 0.3582 - val_accuracy: 0.7898 - val_loss: 0.5746
Epoch 8/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.8737 - loss: 0.3079 - val_accuracy: 0.7982 - v

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# testing on test
network.evaluate(X_test_tfidf, y_test_categorical)

# evaluating before applying the inverse frequency weighting recommendation

predictions = network.predict(X_test_tfidf)
# find max value per output vector
y_pred = np.argmax(predictions, axis=1)
# table output showing per class stats
print(f"{classification_report(y_test_labels, y_pred, target_names=label_vectors.classes_)}")






75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8003 - loss: 0.5301
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
                   precision    recall  f1-score   support

        ATC Issue       0.65      0.73      0.69       275
         Conflict       0.60      0.66      0.63       123
        Deviation       0.74      0.68      0.71       589
Equipment Problem       0.93      0.90      0.92      1301
     Ground Event       0.23      0.40      0.29        35
   Inflight Event       0.51      0.70      0.59        61

         accuracy                           0.80      2384
        macro avg       0.61      0.68      0.64      2384
     weighted avg       0.81      0.80      0.81      2384

